# 08. Функции активации, функции потерь, метрики и оптимизатор

Видео: [8. Функции активации, функции потерь и др.](https://www.youtube.com/watch?v=PARx85j3W2M&list=PLBP4Q3FNSLK2rtGPBsK-aMAetYj-8yg_1&index=9)

## О чём урок

В уроках 4–7 ты готовил данные: <span style="color:#7CDBA0">Dataset</span> читает один пример, <span style="color:#E6C07B">transform</span> превращает картинку в тензор, <span style="color:#6CB6FF">DataLoader</span> собирает батч. Дальше батч нужно отдать модели и как-то её учить.

Автор в видео разбирает, что нужно нейросети **кроме самих слоёв** — «необходимые атрибуты НС». Без любого из них обучение не соберётся:

```text
батч x ─► Linear ─► активация (ReLU) ─► Linear ─► выход сети (логиты / координаты)
                                                      │
                     правильные ответы y ─────────────┤
                                                      ▼
                                        функция потерь → одно число loss
                                                      │
                          loss.backward() → градиенты → оптимизатор (Adam) меняет веса

отдельно: метрика (accuracy и т.п.) — понятная человеку оценка качества
```

| атрибут | на какой вопрос отвечает |
|:---|:---|
| функция активации | как сделать сеть нелинейной, чтобы она выучила что-то сложнее прямой |
| функция потерь | насколько сеть ошиблась — число, которое уменьшаем при обучении |
| метрика | хорошая ли модель — в понятных человеку единицах |
| оптимизатор | как по градиентам поменять веса |
| `nn.Linear` | из чего состоит сама сеть, какие у слоёв размеры и сколько в ней параметров |

Видео обзорное, цикла обучения в нём нет. Модель целиком (`nn.Module`) будет в видео 9–10, цикл обучения — в видео 11. Здесь каждую деталь пробуешь **отдельно на маленьких тензорах**, чтобы потом в модели и цикле они были уже знакомы.

## Где теория в твоих конспектах

Курс Дубинина показывает **как вызвать** это в PyTorch. Формулы и «зачем» — в конспектах и в домашке с линейным классификатором (там softmax+CE руками).

| тема урока 8 | куда смотреть (формулы / смысл) |
|:---|:---|
| зачем loss вообще | [`d.1.2 Loss и эмпирический риск`](../../../все мои конспекты в ynote/d.Машинное обучение/d.1 Введение в мл/d.1.2 Loss и эмпирический риск.md) |
| зачем активации, ReLU / sigmoid / softmax | [`d.4.1`](../../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.1 Полносвязные нейронные сети/d.4.1 Полносвязные нейронные сети.md) **§8–9** |
| голова + loss (MSE / CE / BCE) | [`d.4.1`](../../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.1 Полносвязные нейронные сети/d.4.1 Полносвязные нейронные сети.md) **§10** |
| softmax + CE с расчётами (NumPy) | [`формулы-конспект-код.md`](../../homework_linear_models_and_knn/формулы-конспект-код.md) + [`Linear classifier.ipynb`](../../homework_linear_models_and_knn/Linear%20classifier.ipynb) |
| log loss vs метрика (accuracy, F1) | [`d.3.1 Метрики`](../../../все мои конспекты в ynote/d.Машинное обучение/d.3 Classic ML/d.3.1 Метрики классификации и регрессии.md) |
| градиент / цепочка (как loss тянет веса) | [`d.0`](../../../все мои конспекты в ynote/d.Машинное обучение/d.0 Градиент и правило цепочки.md) |
| forward → loss → backward → шаг | [`d.4.1`](../../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.1 Полносвязные нейронные сети/d.4.1 Полносвязные нейронные сети.md) **§11** |
| SGD, Adam, learning rate | [`d.4.2`](../../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.2 Градиентная оптимизация.md) **§2.2, §2.5, §2.6** |
| полная карта курса ↔ конспекты | [`0_как_учить_pytorch.md`](../0_как_учить_pytorch.md) |

Открывай параграф / домашку рядом с блоком — сначала «зачем» и формулы, потом код PyTorch.


## План

| блок | что делаешь | зачем |
|:---|:---|:---|
| 1 | `ReLU`, `Sigmoid`, `Tanh`; проверяешь, что без активации два слоя = один | понять, зачем нужны нелинейные переходы |
| 2 | `Softmax` на выходе классификатора | превратить выход сети в «уверенность» по классам |
| 3 | `MSELoss`, `L1Loss`, `CrossEntropyLoss`, BCE | выбрать loss под задачу и не ошибиться с форматом входа |
| 4 | accuracy и метрики для регрессии | понять, хороша ли модель, а не только «loss упал» |
| 5 | `Adam`, `SGD` и learning rate | как веса меняются по градиенту и почему важен lr |
| 6 | `nn.Linear`: размеры слоёв, число параметров, ошибка формы | собрать сеть 784 → 128 → 10 из видео |
| 7 | обучаемые параметры против гиперпараметров | что учит сеть, а что выбираешь ты |
| 8 | итог: что сделали и зачем | |

**Как решать.** Замени `...` на код → запусти ячейку → запусти проверку под ней. Сверяй печать с блоком «Ожидаемый вывод». Если проверка упала, сначала читай текст ошибки. Решения — в `08_активации_потери_оптимизатор_решения.ipynb`, только после своей попытки.


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)


def need(cond, msg):
    assert cond, msg


def need_tensor(x, name):
    assert isinstance(x, torch.Tensor), f"{name} — пока не тензор (замени ... на код)"


print("torch", torch.__version__)

torch 2.14.1+cpu


## 1. Функции активации: <span style="color:#C9A0DC">ReLU</span>, <span style="color:#C9A0DC">Sigmoid</span>, <span style="color:#C9A0DC">Tanh</span>

> **Конспект:** [`d.4.1`](../../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.1 Полносвязные нейронные сети/d.4.1 Полносвязные нейронные сети.md) **§8–9**. Скрытый слой ≈ Linear + ReLU; на выходе — «голова» (§10).

### Что это

Линейный слой считает взвешенную сумму входов. Функция активации стоит **между слоями**: берёт выход одного слоя, преобразует **каждое число отдельно** и отдаёт результат следующему слою.

Форму тензора активация не меняет: пришло `(32, 128)` — ушло `(32, 128)`.

### Основные функции из видео

| класс | что делает с числом | диапазон выхода | где применяют |
|:---|:---|:---|:---|
| `nn.ReLU()` | отрицательное → 0, положительное без изменений | от 0 до +∞ | скрытые слои почти всех сетей (полносвязных, свёрточных) — выбор по умолчанию |
| `nn.Sigmoid()` | сжимает любое число в интервал (0, 1) | (0, 1) | выход бинарной классификации, рекуррентные сети |
| `nn.Tanh()` | сжимает любое число в интервал (−1, 1) | (−1, 1) | рекуррентные сети |
| `nn.Softmax(dim)` | превращает вектор в «уверенности», сумма = 1 | (0, 1) | выход многоклассовой классификации, трансформеры — блок 2 |

$$
\text{ReLU}(x) = \max(0, x) \qquad \sigma(x) = \frac{1}{1 + e^{-x}} \qquad \tanh(x) = \frac{e^{x} - e^{-x}}{e^{x} + e^{-x}}
$$

### Как пользоваться

Активация — это класс из `torch.nn`. Сначала создаёшь объект, потом вызываешь его как функцию:

```python
relu = nn.ReLU()   # создать
y = relu(x)        # применить
```

Это тот же приём «создать объект → вызвать», что у transform в уроке 7: `tf = v2.Compose(...)`, потом `tf(img)`.

### Почему в скрытых слоях ReLU, а не сигмоида

У сигмоиды и tanh при больших по модулю x график почти плоский, производная близка к нулю. В глубокой сети такие маленькие производные перемножаются при `backward()`, и градиент до первых слоёв почти не доходит — они не учатся. У ReLU для x > 0 производная всегда равна 1, градиент не затухает. Подробнее — конспект `d.4.1`.

### Задание 1.1

Создай три активации и примени каждую к `h`. `h` — как будто выход первого слоя для одного примера.

**Ожидаемый вывод:**

```text
relu    tensor([0.0000, 0.0000, 0.0000, 0.5000, 2.0000])
sigmoid tensor([0.1190, 0.3780, 0.5000, 0.6220, 0.8810])
tanh    tensor([-0.9640, -0.4620,  0.0000,  0.4620,  0.9640])
```

ReLU обнулила отрицательные значения. Sigmoid(0) = 0.5, tanh(0) = 0. Обе «сжимают» края: 2 и 0.5 у tanh дают 0.964 и 0.462.


In [ ]:
# 1.1 Создай nn.ReLU, nn.Sigmoid, nn.Tanh и примени каждую к h.
h = torch.tensor([-2.0, -0.5, 0.0, 0.5, 2.0])

relu = nn.ReLU()
sigmoid = nn.Sigmoid()
tanh = nn.Tanh()

r = relu(h)
s = sigmoid(h)
t = tanh(h)
print("relu   ", r)
print("sigmoid", s.round(decimals=3))
print("tanh   ", t.round(decimals=3))

relu    tensor([0.0000, 0.0000, 0.0000, 0.5000, 2.0000])
sigmoid tensor([0.1190, 0.3780, 0.5000, 0.6220, 0.8810])
tanh    tensor([-0.9640, -0.4620,  0.0000,  0.4620,  0.9640])


In [ ]:
need(
    isinstance(relu, nn.ReLU) and isinstance(sigmoid, nn.Sigmoid) and isinstance(tanh, nn.Tanh),
    "нужны объекты nn.ReLU(), nn.Sigmoid(), nn.Tanh()",
)
need_tensor(r, "r")
need_tensor(s, "s")
need_tensor(t, "t")
need(torch.equal(r, torch.tensor([0.0, 0.0, 0.0, 0.5, 2.0])), f"ReLU: {r}")
need(bool((s > 0).all() and (s < 1).all()), "sigmoid даёт значения в (0, 1)")
need(abs(float(s[2]) - 0.5) < 1e-6, "sigmoid(0) = 0.5")
need(bool((t > -1).all() and (t < 1).all()) and abs(float(t[2])) < 1e-6, "tanh: (-1, 1), tanh(0) = 0")
need(r.shape == h.shape == s.shape == t.shape, "активация не меняет форму")
print("1.1 — верно")

1.1 — верно


### Задание 1.2 — зачем вообще нелинейность

> **Конспект:** [`d.4.1`](../../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.1 Полносвязные нейронные сети/d.4.1 Полносвязные нейронные сети.md) **§8** — без нелинейности несколько Linear схлопываются в одну линейную модель.

Автор говорит: «нейронная сеть состоит из слоёв с **нелинейными переходами** между ними». Проверим, что будет, если переходы убрать.

Забегая вперёд: `nn.Linear(in, out)` (подробно — блок 6) считает $y = x W^T + b$. Веса лежат в `layer.weight` формы `(out, in)`, смещение — в `layer.bias` формы `(out,)`.

Два линейных слоя подряд без активации:

$$
(x W_1^T + b_1)\, W_2^T + b_2 = x\, (W_2 W_1)^T + (W_2 b_1 + b_2)
$$

Справа снова **один** линейный слой с весами $W = W_2 W_1$ и смещением $b = W_2 b_1 + b_2$. Сколько линейных слоёв ни ставь подряд, сеть останется одним линейным слоем и не выучит ничего сложнее линейной регрессии. Активация между слоями ломает эту цепочку — поэтому её и ставят.

Посчитай `W`, `b`, ответ «одного слоя» `one_layer = xs @ W.T + b` и выход сети с ReLU между слоями `with_relu`.

Подсказка: матричное умножение — `@`, транспонирование — `.T`.

**Ожидаемый вывод:**

```text
W: (2, 4) b: (2,)
без активации = один слой? True
с ReLU = один слой? False
```


In [ ]:
# 1.2 Два Linear без активации = один Linear. С ReLU — уже нет.
torch.manual_seed(0)
lin1 = nn.Linear(4, 3)
lin2 = nn.Linear(3, 2)
xs = torch.randn(5, 4)  # 5 примеров по 4 признака

two_layers = lin2(lin1(xs))  # два линейных слоя подряд, без активации

W = lin2.weight @ lin1.weight  # (2, 3) @ (3, 4) → (2, 4)
b = lin2.weight @ lin1.bias + lin2.bias  # (2, 3) @ (3,) + (2,) → (2,)
one_layer = xs @ W.T + b  # один «эквивалентный» слой
with_relu = lin2(relu(lin1(xs)))  # ReLU между слоями

print("W:", tuple(W.shape), "b:", tuple(b.shape))
print("без активации = один слой?", torch.allclose(two_layers, one_layer, atol=1e-5))
print("с ReLU = один слой?", torch.allclose(with_relu, one_layer, atol=1e-5))

W: (2, 4) b: (2,)
без активации = один слой? True
с ReLU = один слой? False


In [ ]:
need_tensor(W, "W")
need_tensor(b, "b")
need_tensor(one_layer, "one_layer")
need_tensor(with_relu, "with_relu")
need(tuple(W.shape) == (2, 4) and tuple(b.shape) == (2,), f"W {tuple(W.shape)}, b {tuple(b.shape)}")
need(torch.allclose(two_layers, one_layer, atol=1e-5), "без активации два слоя должны совпасть с одним")
need(not torch.allclose(with_relu, one_layer, atol=1e-5), "с ReLU сеть уже не сводится к одному слою")
print("1.2 — верно")

1.2 — верно


## 2. <span style="color:#C9A0DC">Softmax</span> — выход классификатора

> **Конспект / формулы:** [`d.4.1`](../../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.1 Полносвязные нейронные сети/d.4.1 Полносвязные нейронные сети.md) **§9.3**, **§10.2**; расчёты softmax руками — [`формулы-конспект-код.md`](../../homework_linear_models_and_knn/формулы-конспект-код.md) (шаг 1) и домашка `Linear classifier`. В PyTorch при обучении softmax часто **внутри** `CrossEntropyLoss` — см. блок 3.

### Проблема

Последний слой классификатора на FashionMNIST выдаёт 10 чисел на картинку — по одному на класс. Это «сырые» числа любого знака и размера, их называют **логиты** (logits). По ним уже видно, какой класс сеть выбрала (самое большое число), но непонятно, **насколько** она уверена.

### Что делает Softmax

$$
\text{softmax}(z)_i = \frac{e^{z_i}}{\sum_j e^{z_j}}
$$

1. Каждое значение — от 0 до 1.
2. Сумма по всем классам равна 1.
3. Порядок сохраняется: самый большой логит даёт самое большое значение.

Пример из видео: после softmax почти все значения порядка $10^{-14}$…$10^{-20}$, а одно — 0.98. Читаем так: «сеть на 98% уверена, что это этот класс».

Автор подчёркивает: это **не настоящие вероятности**, мы их только так интерпретируем. Сеть может выдать 0.98 и при этом ошибиться.

### Параметр `dim`

Батч логитов имеет форму `(B, C)`: строка — одна картинка, столбец — класс. Softmax нужно считать **внутри каждой строки**, то есть вдоль оси классов: `nn.Softmax(dim=1)`.

Если написать `dim=0`, единице станет равна сумма по **столбцу** — «уверенности» перемешаются между разными картинками. Ошибки при этом не будет, просто ответ станет бессмысленным.

Это тот же смысл `dim`, что у `sum(dim=...)` из урока 3: ось, вдоль которой идёт операция.

### Задание 2.1

Создай `nn.Softmax` по оси классов, получи `probs` и предсказанные классы двумя способами: по логитам и по softmax.

`argmax(dim=1)` — номер самого большого элемента в каждой строке, то есть предсказанный класс.

**Ожидаемый вывод:**

```text
tensor([[0.6380, 0.2350, 0.0950, 0.0320],
        [0.0680, 0.0680, 0.8240, 0.0410]])
суммы по строкам: tensor([1.0000, 1.0000])
класс по логитам: tensor([0, 2]) | по softmax: tensor([0, 2])
```

Класс по логитам и по softmax совпал. Чтобы просто получить ответ сети, softmax не нужен — хватает `argmax` по логитам. Softmax нужен, когда хочешь увидеть уверенность.


In [ ]:
# 2.1 Softmax по оси классов + предсказанный класс
logits = torch.tensor(
    [
        [2.0, 1.0, 0.1, -1.0],  # картинка 0
        [0.5, 0.5, 3.0, 0.0],  # картинка 1
    ]
)  # (B=2, C=4)

softmax = nn.Softmax(dim=1)
probs = softmax(logits)
pred_logits = logits.argmax(dim=1)
pred_probs = probs.argmax(dim=1)

print(probs.round(decimals=3))
print("суммы по строкам:", probs.sum(dim=1))
print("класс по логитам:", pred_logits, "| по softmax:", pred_probs)

tensor([[0.6380, 0.2350, 0.0950, 0.0320],
        [0.0680, 0.0680, 0.8240, 0.0410]])
суммы по строкам: tensor([1.0000, 1.0000])
класс по логитам: tensor([0, 2]) | по softmax: tensor([0, 2])


In [ ]:
need(isinstance(softmax, nn.Softmax), "нужен объект nn.Softmax(...)")
need(softmax.dim == 1, f"softmax по оси классов: dim=1, а сейчас dim={softmax.dim}")
need_tensor(probs, "probs")
need(torch.allclose(probs.sum(dim=1), torch.ones(2)), "сумма в каждой строке = 1")
need(torch.allclose(probs, torch.softmax(logits, dim=1)), "probs не совпадает с softmax по dim=1")
need_tensor(pred_logits, "pred_logits")
need_tensor(pred_probs, "pred_probs")
need(torch.equal(pred_logits, torch.tensor([0, 2])) and torch.equal(pred_logits, pred_probs), "классы 0 и 2")
print("2.1 — верно")

2.1 — верно


## 3. Функции потерь: <span style="color:#6CB6FF">MSELoss</span>, <span style="color:#6CB6FF">L1Loss</span>, <span style="color:#6CB6FF">CrossEntropyLoss</span>, <span style="color:#6CB6FF">BCE</span>

> **Конспект:** что такое loss — [`d.1.2`](../../../все мои конспекты в ynote/d.Машинное обучение/d.1 Введение в мл/d.1.2 Loss и эмпирический риск.md); голова + лосс — [`d.4.1`](../../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.1 Полносвязные нейронные сети/d.4.1 Полносвязные нейронные сети.md) **§10** (MSE §10.3, CE §10.2, BCE §5.2 / §10.1).

### Что это и зачем

Сеть выдала ответы, у нас есть правильные ответы. Функция потерь (loss) сравнивает их и возвращает **одно число** — насколько модель ошиблась. Градиенты при обучении считаются именно от него: `loss.backward()`.

Важная мысль из видео: **по самому значению loss нельзя понять, хорошая модель или нет.** Loss = 3.9 — это хорошо? Неизвестно. Зато точно известно, что при обучении он должен **уменьшаться**, причём на валидационной выборке, а не только на train. Чтобы понять «хорошо или плохо», есть метрики (блок 4).

### Какая функция для какой задачи

| задача | loss в PyTorch | что подаём (выход сети) | что в target |
|:---|:---|:---|:---|
| регрессия (центр квадрата) | `nn.MSELoss()` — средний квадрат ошибки (MSE) | `float`, `(B, 2)` | `float`, `(B, 2)` |
| регрессия | `nn.L1Loss()` — средняя абсолютная ошибка (MAE) | `float` | `float` |
| многоклассовая классификация | `nn.CrossEntropyLoss()` | логиты `(B, C)` | номера классов `int64`, `(B,)` |
| бинарная классификация | `nn.BCELoss()` / `nn.BCEWithLogitsLoss()` | вероятности / логиты `(B,)` | `float` 0.0 или 1.0, `(B,)` |

Автор называет ещё `KLDivLoss` (дивергенция Кульбака–Лейблера, нужна для вариационных автоэнкодеров) и `CTCLoss` (распознавание речи и текста). В этом курсе они не понадобятся.

**Как пользоваться** — так же, как активациями: `loss_fn = nn.MSELoss()`, потом `loss = loss_fn(pred, target)`. Порядок аргументов важен: **сначала предсказание, потом правильный ответ**.

### Задание 3.1 — регрессия: MSE и MAE

$$
\text{MSE} = \frac{1}{n} \sum (\hat{y} - y)^2 \qquad \text{MAE} = \frac{1}{n} \sum |\hat{y} - y|
$$

$\hat{y}$ — предсказание, $y$ — правильный ответ, $n$ — количество всех чисел (здесь 3 примера × 2 координаты = 6).

Это задача квадратов из уроков 6–7: сеть предсказывает центр `[cx, cy]`. Посчитай обе функции через `nn` и руками.

Подсказки: модуль числа — `.abs()`, среднее — `.mean()`, превратить тензор из одного числа в обычное число — `.item()`.

**Ожидаемый вывод:**

```text
MSE 2.2317 | руками 2.2317
MAE 1.0167 | руками 1.0167
```

**В чём разница.** В третьем примере промах по `cy` — 3 пикселя. В MAE он добавляет 3, а в MSE — 9. MSE сильнее наказывает большие промахи. MAE относится ко всем ошибкам одинаково и спокойнее реагирует на выбросы.


In [ ]:
# 3.1 MSELoss и L1Loss: через nn и руками
pred = torch.tensor([[10.2, 51.7], [30.0, 28.5], [44.9, 12.0]])  # (3, 2) предсказанные [cx, cy]
target = torch.tensor([[10.0, 52.0], [32.0, 28.0], [45.0, 15.0]])  # (3, 2) настоящие [cx, cy]

mse = nn.MSELoss()
mae = nn.L1Loss()
loss_mse = mse(pred, target)
loss_mae = mae(pred, target)
manual_mse = ((pred - target) ** 2).mean()
manual_mae = (pred - target).abs().mean()

print(f"MSE {loss_mse.item():.4f} | руками {manual_mse.item():.4f}")
print(f"MAE {loss_mae.item():.4f} | руками {manual_mae.item():.4f}")

MSE 2.2317 | руками 2.2317
MAE 1.0167 | руками 1.0167


In [ ]:
need(isinstance(mse, nn.MSELoss) and isinstance(mae, nn.L1Loss), "нужны nn.MSELoss() и nn.L1Loss()")
for name in ["loss_mse", "loss_mae", "manual_mse", "manual_mae"]:
    need_tensor(globals()[name], name)
need(loss_mse.dim() == 0, "loss — одно число (тензор без осей)")
need(torch.allclose(loss_mse, manual_mse) and abs(loss_mse.item() - 2.2317) < 1e-3, f"MSE={loss_mse.item():.4f}")
need(torch.allclose(loss_mae, manual_mae) and abs(loss_mae.item() - 1.0167) < 1e-3, f"MAE={loss_mae.item():.4f}")
print("3.1 — верно")

3.1 — верно


### Задание 3.2 — <span style="color:#6CB6FF">CrossEntropyLoss</span>

> **Конспект / формулы:** [`d.4.1`](../../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.1 Полносвязные нейронные сети/d.4.1 Полносвязные нейронные сети.md) **§10.2** (CE = −log p правильного класса). Подробные расчёты softmax+CE и градиент — [`формулы-конспект-код.md`](../../homework_linear_models_and_knn/формулы-конспект-код.md) (шаги 1–3) и [`Linear classifier.ipynb`](../../homework_linear_models_and_knn/Linear%20classifier.ipynb) — там же идея, что ты уже делал руками; здесь `nn.CrossEntropyLoss` делает то же.

Два правила, на которых ошибаются чаще всего:

1. **На вход — логиты, а не softmax.** `CrossEntropyLoss` внутри сама делает softmax. Если подать ей уже готовый softmax, он применится **второй раз**, и loss посчитается неправильно. Ошибки Python при этом не будет — сеть просто будет учиться хуже.
2. **Target — номера классов**: тензор `int64` формы `(B,)`, например `tensor([0, 2])`. Не one-hot.

**Что она считает.** Для каждого примера берёт уверенность сети в **правильном** классе $p$ и считает $-\log p$, потом усредняет по батчу:

$$
\text{CE} = -\frac{1}{B} \sum_{i} \log p_{i,\, y_i}
$$

Если сеть уверена в правильном классе ($p \to 1$), то $-\log p \to 0$ — штрафа почти нет. Если в правильный класс она почти не верит ($p \to 0$), loss большой.

**Отсюда важное следствие.** Автор говорит, что softmax ставят после последнего слоя классификатора. В PyTorch при обучении с `CrossEntropyLoss` **последний слой отдаёт логиты без softmax** — softmax уже внутри loss. Отдельно softmax добавляют, только когда нужно показать уверенность (как в блоке 2).

Посчитай:
- `loss_ce` — правильно, по логитам;
- `manual_ce` — руками по `probs` из 2.1;
- `wrong` — с ошибкой: подай в `ce` не логиты, а `probs`.

Подсказка: `probs[torch.arange(2), y]` — в строке 0 берёт элемент из столбца `y[0]`, в строке 1 — из столбца `y[1]`. То есть уверенность в правильном классе для каждой картинки. Логарифм — `torch.log`.

**Ожидаемый вывод:**

```text
уверенность в правильном классе: tensor([0.6380, 0.8240])
CE по логитам 0.3216 | руками 0.3216
CE по softmax (ошибка) 0.9508
```

`wrong` получился другим числом, хотя данные те же. Это и есть тихая ошибка «softmax перед `CrossEntropyLoss`».


In [ ]:
# 3.2 CrossEntropyLoss: логиты на вход, номера классов в target
y = torch.tensor([0, 2])  # правильные классы: картинка 0 — класс 0, картинка 1 — класс 2

ce = nn.CrossEntropyLoss()
loss_ce = ce(logits, y)  # правильно: логиты
manual_ce = -torch.log(probs[torch.arange(2), y]).mean()  # -log(уверенность в правильном классе)
wrong = ce(probs, y)  # ошибка: softmax применится второй раз

print("уверенность в правильном классе:", probs[torch.arange(2), y].round(decimals=3))
print(f"CE по логитам {loss_ce.item():.4f} | руками {manual_ce.item():.4f}")
print(f"CE по softmax (ошибка) {wrong.item():.4f}")

уверенность в правильном классе: tensor([0.6380, 0.8240])
CE по логитам 0.3216 | руками 0.3216
CE по softmax (ошибка) 0.9508


In [ ]:
need(isinstance(ce, nn.CrossEntropyLoss), "нужен nn.CrossEntropyLoss()")
need(y.dtype == torch.int64, "target для CrossEntropyLoss — int64")
for name in ["loss_ce", "manual_ce", "wrong"]:
    need_tensor(globals()[name], name)
need(torch.allclose(loss_ce, nn.functional.cross_entropy(logits, y)), "loss_ce считается по логитам")
need(torch.allclose(loss_ce, manual_ce, atol=1e-5), "ручной расчёт не совпал с CrossEntropyLoss")
need(abs(wrong.item() - loss_ce.item()) > 0.1, "wrong должен считаться по probs — и отличаться")
print("3.2 — верно")

3.2 — верно


### Задание 3.3 — бинарная классификация: BCE

> **Конспект:** [`d.4.1`](../../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.1 Полносвязные нейронные сети/d.4.1 Полносвязные нейронные сети.md) **§5.2**, **§10.1**; log loss в таблице loss — [`d.1.2`](../../../все мои конспекты в ynote/d.Машинное обучение/d.1 Введение в мл/d.1.2 Loss и эмпирический риск.md). `BCEWithLogitsLoss` = sigmoid + BCE в одном.

Бинарная задача — два класса: больной / здоровый, клик / не клик. Сеть выдаёт **одно число** на пример. Sigmoid переводит его в (0, 1) — уверенность в классе 1.

$$
\text{BCE} = -\frac{1}{n} \sum \big[\, y \log p + (1 - y) \log(1 - p) \,\big]
$$

Если правильный ответ $y = 1$, работает первое слагаемое (штраф за маленькое $p$). Если $y = 0$ — второе (штраф за большое $p$).

В PyTorch два варианта:

| вариант | что подаём | кто делает sigmoid |
|:---|:---|:---|
| `nn.BCELoss()` | вероятности `sigmoid(z)` | ты сам, перед loss |
| `nn.BCEWithLogitsLoss()` | логиты `z` | сам loss, внутри; так считается точнее, на практике пишут его |

Принцип тот же, что у `CrossEntropyLoss`: вариант «с логитами» сам делает нормировку. Target — `float` (0.0 / 1.0), не `int`.

Посчитай оба варианта. `sigmoid` уже создан в 1.1.

**Ожидаемый вывод:**

```text
уверенность в классе 1: tensor([0.8180, 0.4260, 0.9000, 0.1190])
BCELoss(sigmoid(z)) 0.7469 | BCEWithLogitsLoss(z) 0.7469
```

Числа совпали: это одна и та же функция, разница только в том, где применяется sigmoid.


In [ ]:
# 3.3 BCELoss(sigmoid(z)) и BCEWithLogitsLoss(z)
z = torch.tensor([1.5, -0.3, 2.2, -2.0])  # логиты: по одному числу на пример
t_bin = torch.tensor([1.0, 0.0, 1.0, 1.0])  # правильные ответы 0/1, float

bce = nn.BCELoss()
bce_logits = nn.BCEWithLogitsLoss()
loss_bce = bce(sigmoid(z), t_bin)
loss_bce_logits = bce_logits(z, t_bin)

print("уверенность в классе 1:", sigmoid(z).round(decimals=3))
print(f"BCELoss(sigmoid(z)) {loss_bce.item():.4f} | BCEWithLogitsLoss(z) {loss_bce_logits.item():.4f}")

уверенность в классе 1: tensor([0.8180, 0.4260, 0.9000, 0.1190])
BCELoss(sigmoid(z)) 0.7469 | BCEWithLogitsLoss(z) 0.7469


In [ ]:
need(isinstance(bce, nn.BCELoss) and isinstance(bce_logits, nn.BCEWithLogitsLoss), "нужны nn.BCELoss() и nn.BCEWithLogitsLoss()")
need_tensor(loss_bce, "loss_bce")
need_tensor(loss_bce_logits, "loss_bce_logits")
need(torch.allclose(loss_bce, loss_bce_logits, atol=1e-5), "оба варианта должны совпасть")
need(abs(loss_bce_logits.item() - nn.functional.binary_cross_entropy_with_logits(z, t_bin).item()) < 1e-6, "BCEWithLogitsLoss получает логиты z")
print("3.3 — верно")

3.3 — верно


## 4. Метрики: <span style="color:#E6C07B">accuracy</span> и метрики для регрессии

> **Конспект:** loss vs метрика — [`d.3.1`](../../../все мои конспекты в ynote/d.Машинное обучение/d.3 Classic ML/d.3.1 Метрики классификации и регрессии.md) и [`d.1.2`](../../../все мои конспекты в ynote/d.Машинное обучение/d.1 Введение в мл/d.1.2 Loss и эмпирический риск.md); loss для оптимизатора — [`d.4.1`](../../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.1 Полносвязные нейронные сети/d.4.1 Полносвязные нейронные сети.md) **§11**.

### Зачем, если уже есть loss

Loss нужен **сети** — по нему считаются градиенты. Человеку число 0.37 ничего не говорит. **Метрика** — критерий качества, понятный человеку и привязанный к задаче. Для каждой задачи подбирают свою метрику, и подобрать хорошую — отдельная непростая задача («со звёздочкой», как говорит автор).

### Accuracy — доля правильных ответов

$$
\text{accuracy} = \frac{\text{число правильных ответов}}{\text{число всех ответов}}
$$

Пример из видео: на MNIST 60 000 тренировочных картинок, модель угадала 55 000. Accuracy = 55 000 / 60 000 ≈ 0.92: из 100 картинок примерно 92 классифицируются правильно. Это число уже что-то говорит о модели, в отличие от loss.

**Как считать в PyTorch:** логиты → `argmax(dim=1)` → сравнить с правильными ответами → посчитать совпадения.

### Задание 4.1

Подсказки: `pred == y` даёт тензор из `True` / `False`; `.sum()` считает `True`; `.item()` превращает результат в обычное число Python; `len(y_b)` — сколько всего примеров.

**Ожидаемый вывод:**

```text
предсказания: tensor([0, 1, 2, 0, 1, 2, 0, 0])
правильные:   tensor([0, 1, 2, 1, 1, 2, 0, 2])
верно 6 из 8 → accuracy 0.75
```

Ошибки в примерах 3 и 7: там самый большой логит не у правильного класса.


In [ ]:
# 4.1 accuracy: логиты → argmax → сравнить с y
logits_b = torch.tensor(
    [
        [2.0, 0.1, -1.0],
        [0.2, 1.5, 0.3],
        [0.1, 0.2, 2.5],
        [1.2, 1.0, 0.0],
        [-0.5, 0.3, 0.1],
        [0.0, 0.0, 3.0],
        [2.2, 1.9, 0.4],
        [0.3, 0.1, 0.2],
    ]
)  # (8, 3): 8 картинок, 3 класса
y_b = torch.tensor([0, 1, 2, 1, 1, 2, 0, 2])

pred_b = logits_b.argmax(dim=1)
correct = (pred_b == y_b).sum().item()
acc = correct / len(y_b)

print("предсказания:", pred_b)
print("правильные:  ", y_b)
print(f"верно {correct} из {len(y_b)} → accuracy {acc:.2f}")

предсказания: tensor([0, 1, 2, 0, 1, 2, 0, 0])
правильные:   tensor([0, 1, 2, 1, 1, 2, 0, 2])
верно 6 из 8 → accuracy 0.75


In [ ]:
need_tensor(pred_b, "pred_b")
need(torch.equal(pred_b, torch.tensor([0, 1, 2, 0, 1, 2, 0, 0])), f"pred_b={pred_b}")
need(isinstance(correct, int) and correct == 6, f"correct={correct!r}, нужно обычное число 6 (.item())")
need(abs(acc - 0.75) < 1e-9, f"acc={acc}")
print("4.1 — верно")

4.1 — верно


### Задание 4.2 — где accuracy обманывает

Минус accuracy из видео: она **не учитывает цену ошибки**. Модель классифицирует людей на здоровых и больных:
- здорового приняли за больного — неприятно, но не страшно: его дообследуют;
- больного приняли за здорового — это может стоить пациенту здоровья.

Для accuracy обе ошибки весят одинаково.

Вторая ловушка — **несбалансированные классы**. Если больных мало, «ленивая» модель, которая всем отвечает «здоров», получает высокую accuracy и при этом не находит ни одного больного.

Пусть из 100 человек больны 5 (класс 1), а модель всем отвечает 0. Посчитай её accuracy и сколько больных она нашла.

Подсказки: `.float().mean()` — доля `True`; `&` — поэлементное «и» для тензоров из `True` / `False`, скобки вокруг сравнений обязательны: `(a == 1) & (b == 1)`.

**Ожидаемый вывод:**

```text
accuracy 0.95 | найдено больных 0 из 5
```

Для FashionMNIST accuracy подходит: классы сбалансированы (по 6000 картинок на класс), и цена ошибки одинаковая — перепутать рубашку с футболкой не хуже, чем кроссовок с сандалией.


In [ ]:
# 4.2 «Ленивая» модель на несбалансированных данных
y_med = torch.cat([torch.zeros(95), torch.ones(5)]).long()  # 95 здоровых (0), 5 больных (1)
pred_lazy = torch.zeros(100, dtype=torch.long)  # всем ответила «здоров»

acc_lazy = (pred_lazy == y_med).float().mean().item()
found_sick = ((pred_lazy == 1) & (y_med == 1)).sum().item()

print(f"accuracy {acc_lazy:.2f} | найдено больных {found_sick} из 5")

accuracy 0.95 | найдено больных 0 из 5


In [ ]:
need(isinstance(acc_lazy, float) and abs(acc_lazy - 0.95) < 1e-6, f"acc_lazy={acc_lazy!r}")
need(isinstance(found_sick, int) and found_sick == 0, f"found_sick={found_sick!r}")
print("4.2 — верно")

4.2 — верно


### Задание 4.3 — метрика для регрессии (центр квадрата)

> **Конспект:** MSE — [`d.4.1`](../../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.1 Полносвязные нейронные сети/d.4.1 Полносвязные нейронные сети.md) **§5.1**, **§10.3**; squared/absolute в [`d.1.2`](../../../все мои конспекты в ynote/d.Машинное обучение/d.1 Введение в мл/d.1.2 Loss и эмпирический риск.md). Метрика «радиус» — поверх MSE.

Accuracy в лоб тут не работает: сеть почти никогда не выдаст **ровно** 10.0 и 52.0, поэтому сравнение «равно» почти всегда даст `False`. Автор предлагает два варианта — и вообще метрику для регрессии можно придумать любую, под свою задачу:

| вариант | правило «ответ правильный» |
|:---|:---|
| округление | округлить предсказание до целых и сравнить с ответом; должны совпасть **обе** координаты |
| радиус | расстояние от предсказания до настоящего центра ≤ R пикселей |

Расстояние между точками:

$$
d = \sqrt{(\hat{c}_x - c_x)^2 + (\hat{c}_y - c_y)^2}
$$

Подсказки: `.round()` — округление; `.all(dim=1)` — `True`, если в строке все значения `True`; `.sum(dim=1)` — сумма по строке; `.sqrt()` — корень; доля `True` — `.float().mean()`.

**Ожидаемый вывод:**

```text
расстояния: tensor([0.5700, 1.0800, 3.0000, 0.4500])
по округлению: tensor([ True, False, False,  True]) доля 0.50
по радиусу 2:   tensor([ True,  True, False,  True]) доля 0.75
```

Во втором примере промах на 1 пиксель по `cx`: округление считает это ошибкой, радиус — засчитывает. Какой вариант брать, решаешь ты по смыслу задачи.


In [ ]:
# 4.3 Две метрики для регрессии: округление и радиус
pred_c = torch.tensor([[10.4, 51.6], [31.0, 28.4], [44.9, 12.0], [20.2, 20.6]])  # (4, 2)
true_c = torch.tensor([[10.0, 52.0], [32.0, 28.0], [45.0, 15.0], [20.0, 21.0]])  # (4, 2)
R = 2.0

hit_round = (pred_c.round() == true_c).all(dim=1)  # (4,) True/False
dist = ((pred_c - true_c) ** 2).sum(dim=1).sqrt()  # (4,) расстояние до центра
hit_radius = dist <= R  # (4,) True/False

print("расстояния:", dist.round(decimals=2))
print("по округлению:", hit_round, f"доля {hit_round.float().mean().item():.2f}")
print(f"по радиусу {R:g}:  ", hit_radius, f"доля {hit_radius.float().mean().item():.2f}")

расстояния: tensor([0.5700, 1.0800, 3.0000, 0.4500])
по округлению: tensor([ True, False, False,  True]) доля 0.50
по радиусу 2:   tensor([ True,  True, False,  True]) доля 0.75


In [ ]:
for name in ["hit_round", "dist", "hit_radius"]:
    need_tensor(globals()[name], name)
need(tuple(hit_round.shape) == (4,) and hit_round.dtype == torch.bool, "hit_round: (4,) bool — .all(dim=1)")
need(torch.equal(hit_round, torch.tensor([True, False, False, True])), f"hit_round={hit_round}")
need(torch.allclose(dist, torch.tensor([0.5657, 1.0770, 3.0017, 0.4472]), atol=1e-3), f"dist={dist}")
need(torch.equal(hit_radius, torch.tensor([True, True, False, True])), f"hit_radius={hit_radius}")
print("4.3 — верно")

4.3 — верно


## 5. Оптимизатор: <span style="color:#E6C07B">Adam</span>, <span style="color:#E6C07B">SGD</span> и learning rate

> **Конспект:** петля — [`d.4.1`](../../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.1 Полносвязные нейронные сети/d.4.1 Полносвязные нейронные сети.md) **§11**; цепочка/градиент — [`d.0`](../../../все мои конспекты в ynote/d.Машинное обучение/d.0 Градиент и правило цепочки.md); SGD/Adam/lr — [`d.4.2`](../../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.2 Градиентная оптимизация.md) **§2.2, §2.5, §2.6**.

### Что это

При обучении `loss.backward()` считает градиенты для всех весов — это ты делал в уроке 3 с `requires_grad`. Дальше веса нужно сдвинуть **против градиента**, туда, где loss меньше. Это градиентный спуск.

**Оптимизатор** — объект, который делает этот сдвиг. Adam, RMSprop и другие — улучшенные версии обычного градиентного спуска (SGD): быстрее сходятся к минимуму и реже застревают в локальных минимумах.

| оптимизатор | класс в PyTorch |
|:---|:---|
| SGD | `torch.optim.SGD` |
| RMSprop | `torch.optim.RMSprop` |
| Adam | `torch.optim.Adam` — автор дальше почти всегда берёт его |

При создании передают **что учить** и **с какой скоростью**:

```python
opt = torch.optim.Adam(model.parameters(), lr=0.001)
```

Один шаг обучения — всегда три строки, в таком порядке:

```python
opt.zero_grad()   # обнулить старые градиенты (иначе они накапливаются — урок 3)
loss.backward()   # посчитать новые градиенты
opt.step()        # сдвинуть веса
```

Цикл обучения целиком — видео 11. Здесь пробуем на одном числе вместо модели.

### Задание 5.1 — один шаг Adam

Минимизируем $(w - 3)^2$ — минимум при $w = 3$. Модели пока нет, поэтому вместо `model.parameters()` передаём список из одного тензора: `[w]`.

Градиент в точке $w = 0$: $2(w - 3) = -6$. Минус значит «двигайся вправо, к 3».

**Ожидаемый вывод:**

```text
градиент: -6.0 | w после шага: 0.001
```

Почему сдвиг ровно 0.001: на первом шаге Adam сдвигает каждый вес примерно на `lr` в сторону уменьшения loss, независимо от величины градиента. Поэтому у Adam `lr` — это примерно «размер шага».


In [ ]:
# 5.1 Один шаг Adam для w, минимизируем (w - 3)^2
w = torch.tensor([0.0], requires_grad=True)
opt = torch.optim.Adam([w], lr=0.001)

loss_w = ((w - 3) ** 2).sum()
opt.zero_grad()
loss_w.backward()
opt.step()

print("градиент:", w.grad.item(), "| w после шага:", round(w.item(), 6))

градиент: -6.0 | w после шага: 0.001


In [ ]:
need(isinstance(opt, torch.optim.Adam), "нужен torch.optim.Adam([w], lr=0.001)")
need(opt.param_groups[0]["lr"] == 0.001, f"lr={opt.param_groups[0]['lr']}")
need(w.grad is not None and abs(w.grad.item() + 6.0) < 1e-6, "градиент должен быть -6: забыл backward()?")
need(abs(w.item() - 0.001) < 1e-6, f"w={w.item()}: после step() должно стать 0.001")
print("5.1 — верно")

5.1 — верно


### Задание 5.2 — learning rate: мало, нормально, много

> **Конспект:** [`d.4.2 Градиентная оптимизация`](../../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.2 Градиентная оптимизация.md) **§2.6** — таблица/график: слишком большой α скачет или расходится, слишком маленький — еле ползёт.

Что говорит автор про скорость обучения (`lr`):
- **большой lr** — loss сначала быстро падает, но недолго: дальше значения скачут вокруг одного числа, то больше, то меньше;
- **маленький lr** — loss падает медленно, но уверенно, и с большой вероятностью застрянет в локальном минимуме, а нужен глобальный;
- стартовое значение для Adam — **0.001** (рекомендация, не строгое правило), а по ходу обучения lr **понижают** — автоматически это делается в видео 12 (LR_Scheduler).

Посмотрим на SGD: у него шаг = `lr × градиент`, поэтому эффект виден нагляднее, чем у Adam. Допиши три строки шага внутри функции — те же, что в 5.1.

**Ожидаемый вывод:**

```text
lr=0.001 первые шаги [0.006, 0.012, 0.018, 0.024]  после 20: 0.118
lr=0.1   первые шаги [0.6, 1.08, 1.464, 1.771]  после 20: 2.965
lr=0.9   первые шаги [5.4, 1.08, 4.536, 1.771]  после 20: 2.965
lr=1.1   первые шаги [6.6, -1.32, 8.184, -3.221]  после 20: -112.013
```

Как читать:
- **0.001** — идём в правильную сторону, но за 20 шагов почти не сдвинулись;
- **0.1** — спокойно подходим к 3;
- **0.9** — перепрыгиваем 3 то справа, то слева (5.4 → 1.08 → 4.536 …). Это и есть «скачет». Здесь функция простая, поэтому в итоге всё-таки сошлись к тому же 2.965, что и при 0.1, но путь рваный; в настоящей сети такие скачки видны как loss, который прыгает вверх-вниз;
- **1.1** — каждый прыжок больше предыдущего, `w` улетает. Обучение разошлось: в реальной сети loss растёт или становится `nan`.


In [ ]:
# 5.2 SGD с разными lr на (w - 3)^2: 20 шагов из w = 0
def run_sgd(lr, steps=20):
    w = torch.tensor([0.0], requires_grad=True)
    opt = torch.optim.SGD([w], lr=lr)
    history = []
    for _ in range(steps):
        loss = ((w - 3) ** 2).sum()
        opt.zero_grad()
        loss.backward()
        opt.step()
        history.append(w.item())
    return history


for lr in [0.001, 0.1, 0.9, 1.1]:
    hist = run_sgd(lr)
    first = [round(v, 3) for v in hist[:4]]
    print(f"lr={lr:<5} первые шаги {first}  после 20: {hist[-1]:.3f}")

lr=0.001 первые шаги [0.006, 0.012, 0.018, 0.024]  после 20: 0.118
lr=0.1   первые шаги [0.6, 1.08, 1.464, 1.771]  после 20: 2.965
lr=0.9   первые шаги [5.4, 1.08, 4.536, 1.771]  после 20: 2.965
lr=1.1   первые шаги [6.6, -1.32, 8.184, -3.221]  после 20: -112.013


In [ ]:
h_small, h_ok, h_jump, h_big = (run_sgd(lr) for lr in [0.001, 0.1, 0.9, 1.1])
need(h_ok[0] != 0.0, "w не двигается: проверь zero_grad / backward / step внутри цикла")
need(abs(h_ok[-1] - 3) < 0.05, f"lr=0.1 должен подойти к 3, а w={h_ok[-1]:.3f}")
need(h_small[-1] < 0.5, f"lr=0.001 почти не двигается, а w={h_small[-1]:.3f}")
need(h_jump[0] > 3 and h_jump[1] < 3, "lr=0.9 перепрыгивает 3 туда-обратно")
need(abs(h_big[-1] - 3) > 10, "lr=1.1 должен разойтись")
print("5.2 — верно")

5.2 — верно


## 6. <span style="color:#7CDBA0">nn.Linear</span>: размеры слоёв и число параметров

> **Конспект:** один нейрон / Linear — [`d.4.1 Полносвязные нейронные сети`](../../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.1 Полносвязные нейронные сети/d.4.1 Полносвязные нейронные сети.md) **§5**; несколько нейронов = слой — **§6**; несколько блоков подряд — **§7**. Картинка → 784 → скрытый → 10 классов — схема как в **§7.Б**.

### Что это

`nn.Linear(in_features, out_features)` — **полносвязный** слой: каждый вход связан с каждым нейроном слоя.
- `in_features` — размер входного вектора **одного** примера;
- `out_features` — размер выхода, он же **количество нейронов** в слое.

Батч проходит целиком: вход `(B, in)` → выход `(B, out)`. Размер батча `B` слою не важен, он может быть любым.

### Как подобрать размеры (сеть из видео: 784 → 128 → 10)

1. **Вход и выход задаются задачей.** Картинка MNIST / FashionMNIST `(1, 28, 28)` вытягивается в вектор из 28 · 28 = 784 чисел. Классов 10 — значит, на выходе 10 чисел.
2. **Скрытый слой — твой выбор.** Автор берёт 128 нейронов. Слишком много — сеть тяжёлая и быстро переобучается на простых данных. Слишком мало — не сможет обучиться. Что значит «много» или «мало», подскажут только опыт и эксперименты.
3. **Правило стыковки:** `out_features` одного слоя = `in_features` следующего. Активация между ними размер не меняет: пришло 128 — ушло 128.

```text
(B, 1, 28, 28) → reshape → (B, 784) → Linear(784, 128) → (B, 128) → ReLU → (B, 128) → Linear(128, 10) → (B, 10)
```

**Почему здесь можно `reshape`,** хотя в уроке 7 он «ломал» каналы. Там нужно было переставить оси (HWC → CHW), а `reshape` этого не умеет. Здесь мы ничего не переставляем, а просто вытягиваем картинку в одну строку. Порядок пикселей в строке сети не важен — лишь бы он был одинаковым у всех картинок, а `reshape` это гарантирует.

### Задание 6.1

Прогони батч из 32 «картинок» через сеть из видео и напечатай форму на каждом шаге.

Подсказка: `imgs.reshape(imgs.shape[0], -1)` — `-1` значит «посчитай сам» (получится 28 · 28 = 784).

**Ожидаемый вывод:**

```text
torch.Size([32, 1, 28, 28]) torch.Size([32, 784]) torch.Size([32, 128]) torch.Size([32, 128]) torch.Size([32, 10])
```


In [ ]:
# 6.1 Сеть из видео: 784 → 128 → ReLU → 10
torch.manual_seed(0)
imgs = torch.randn(32, 1, 28, 28)  # батч, как из DataLoader для FashionMNIST

x_flat = imgs.reshape(imgs.shape[0], -1)  # (32, 784)
layer_1 = nn.Linear(784, 128)
act = nn.ReLU()
layer_2 = nn.Linear(128, 10)

h1 = layer_1(x_flat)  # (32, 128)
a1 = act(h1)  # (32, 128)
out = layer_2(a1)  # (32, 10)

print(imgs.shape, x_flat.shape, h1.shape, a1.shape, out.shape)

torch.Size([32, 1, 28, 28]) torch.Size([32, 784]) torch.Size([32, 128]) torch.Size([32, 128]) torch.Size([32, 10])


In [ ]:
for name in ["x_flat", "h1", "a1", "out"]:
    need_tensor(globals()[name], name)
need(isinstance(layer_1, nn.Linear) and (layer_1.in_features, layer_1.out_features) == (784, 128), "layer_1 = nn.Linear(784, 128)")
need(isinstance(act, nn.ReLU), "act = nn.ReLU()")
need(isinstance(layer_2, nn.Linear) and (layer_2.in_features, layer_2.out_features) == (128, 10), "layer_2 = nn.Linear(128, 10)")
need(tuple(x_flat.shape) == (32, 784), f"x_flat {tuple(x_flat.shape)}")
need(tuple(h1.shape) == (32, 128) and tuple(a1.shape) == (32, 128), "после layer_1 и ReLU — (32, 128)")
need(bool((a1 >= 0).all()), "a1 — после ReLU, отрицательных быть не должно")
need(tuple(out.shape) == (32, 10), f"out {tuple(out.shape)}")
print("6.1 — верно")

6.1 — верно


### Задание 6.2 — сколько у сети обучаемых параметров

Каждая связь «вход → нейрон» — это один вес. Кроме того, у каждого нейрона есть своё **смещение** (bias). Оно добавляется автоматически, указывать его не нужно (отключить можно через `bias=False`).

$$
\text{параметров слоя} = \text{in} \cdot \text{out} + \text{out}
$$

Первый слой из видео: 784 · 128 + 128 = 100 480 — «чуть больше 100 тысяч». Это минус полносвязных слоёв: параметров очень много, особенно по сравнению со свёрточными (видео 17).

Второй слой автор оставил посчитать самостоятельно. Посчитай его руками и проверь через PyTorch.

Подсказки:
- `layer.parameters()` — все обучаемые тензоры слоя (`weight` и `bias`);
- `p.numel()` — сколько чисел в тензоре `p`;
- `sum(p.numel() for p in layer.parameters())` — всего параметров в слое.

Обрати внимание на форму `layer_1.weight`: `(out, in)` = `(128, 784)`, а не `(784, 128)`. Поэтому `nn.Linear` и считает $x W^T$, с транспонированием.

**Ожидаемый вывод:**

```text
layer_1.weight: torch.Size([128, 784]) | layer_1.bias: torch.Size([128])
layer_1: руками 100480 | PyTorch 100480
layer_2: руками 1290 | PyTorch 1290
ReLU: 0 | вся сеть: 101770
```

У ReLU параметров нет: это просто функция, учить в ней нечего.


In [ ]:
# 6.2 Число обучаемых параметров: руками и через PyTorch
n_layer_1_manual = 784 * 128 + 128
n_layer_2_manual = 128 * 10 + 10

n_layer_1 = sum(p.numel() for p in layer_1.parameters())
n_layer_2 = sum(p.numel() for p in layer_2.parameters())
n_act = sum(p.numel() for p in act.parameters())
n_total = n_layer_1 + n_act + n_layer_2

print("layer_1.weight:", layer_1.weight.shape, "| layer_1.bias:", layer_1.bias.shape)
print(f"layer_1: руками {n_layer_1_manual} | PyTorch {n_layer_1}")
print(f"layer_2: руками {n_layer_2_manual} | PyTorch {n_layer_2}")
print(f"ReLU: {n_act} | вся сеть: {n_total}")

layer_1.weight: torch.Size([128, 784]) | layer_1.bias: torch.Size([128])
layer_1: руками 100480 | PyTorch 100480
layer_2: руками 1290 | PyTorch 1290
ReLU: 0 | вся сеть: 101770


In [ ]:
need(n_layer_2_manual == 1290, f"второй слой руками: 128 * 10 + 10, а у тебя {n_layer_2_manual!r}")
need(n_layer_1 == 100480 and n_layer_2 == 1290, f"через PyTorch: layer_1={n_layer_1!r}, layer_2={n_layer_2!r}")
need(n_act == 0, "у ReLU нет параметров")
need(n_total == 101770, f"вся сеть: {n_total!r}")
print("6.2 — верно")

6.2 — верно


### Задание 6.3 — ошибка формы (shape mismatch)

Самая частая ошибка при сборке сети — не состыковать размеры слоёв. Вот как она выглядит:

```text
mat1 and mat2 shapes cannot be multiplied (32x128 and 64x10)
```

Как читать:
- `mat1` = то, что пришло в слой: 32 примера по **128** чисел;
- `mat2` = веса слоя (транспонированные): слой ждёт **64** числа на входе и выдаёт 10;
- 128 ≠ 64 — значит, `in_features` этого слоя не совпадает с `out_features` предыдущего.

Ячейка ниже ловит ошибку и печатает её текст. Создай исправленный слой `fixed_layer_2`, который примет `a1`.

**Ожидаемый вывод:**

```text
ошибка: mat1 and mat2 shapes cannot be multiplied (32x128 and 64x10)
после исправления: torch.Size([32, 10])
```


In [ ]:
# 6.3 Неправильный in_features → ошибка. Исправь второй слой.
bad_layer_2 = nn.Linear(64, 10)  # ошибка: ждёт 64 числа, а приходит 128
try:
    bad_layer_2(a1)
except RuntimeError as e:
    print("ошибка:", e)

fixed_layer_2 = nn.Linear(128, 10)
print("после исправления:", fixed_layer_2(a1).shape)

ошибка: mat1 and mat2 shapes cannot be multiplied (32x128 and 64x10)
после исправления: torch.Size([32, 10])


In [ ]:
need(isinstance(fixed_layer_2, nn.Linear), "fixed_layer_2 — это nn.Linear(...)")
need(fixed_layer_2.in_features == a1.shape[1], f"in_features должен быть {a1.shape[1]}")
need(tuple(fixed_layer_2(a1).shape) == (32, 10), "на выходе (32, 10)")
print("6.3 — верно")

6.3 — верно


## 7. Обучаемые параметры и гиперпараметры

> **Конспект:** веса W, b учит оптимизатор ([§11](../../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.1 Полносвязные нейронные сети/d.4.1 Полносвязные нейронные сети.md), [`d.4.2 Градиентная оптимизация`](../../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.2 Градиентная оптимизация.md)). Число слоёв, lr, размер батча — гиперпараметры: их задаёшь ты до обучения (см. также эксперименты в `0_как_учить_pytorch.md`).

| | кто задаёт | примеры |
|:---|:---|:---|
| **обучаемые параметры** | сеть подбирает сама во время обучения (через оптимизатор) | веса и смещения всех слоёв — те 101 770 чисел из 6.2 |
| **гиперпараметры** | ты, **до** начала обучения | число слоёв, число нейронов в слое, learning rate, доли train / val / test, функция активации, функция потерь, число эпох, размер батча |

Автор: к гиперпараметрам можно отнести всё, что относится к нейросети и влияет на её обучение, но само не обучается. Подбор гиперпараметров — это и есть эксперименты: меняешь одну вещь за запуск и сравниваешь результат на val.

### Задание 7.1

Заполни словарь: для каждого пункта напиши `"обучаемый"` или `"гипер"`.

**Ожидаемый вывод:**

```text
обучаемых: 2 | гипер: 6
```


In [ ]:
# 7.1 Для каждого пункта: "обучаемый" или "гипер"
kinds = {
    "layer_1.weight": "обучаемый",
    "layer_2.bias": "обучаемый",
    "lr = 0.001": "гипер",
    "128 нейронов в скрытом слое": "гипер",
    "batch_size = 32": "гипер",
    "ReLU как функция активации": "гипер",
    "число эпох": "гипер",
    "доли train / val / test": "гипер",
}

values = list(kinds.values())
print("обучаемых:", values.count("обучаемый"), "| гипер:", values.count("гипер"))

обучаемых: 2 | гипер: 6


In [ ]:
trainable = {"layer_1.weight", "layer_2.bias"}
wrong_keys = [k for k, v in kinds.items() if v != ("обучаемый" if k in trainable else "гипер")]
need(not wrong_keys, f"проверь пункты: {wrong_keys}")
print("7.1 — верно")

7.1 — верно


## 8. Что мы сделали в этом уроке

1. **Разобрали функции активации.** `ReLU` — по умолчанию для скрытых слоёв, `Sigmoid` и `Tanh` сжимают значения в (0, 1) и (−1, 1). Проверили, что без активации два линейных слоя сводятся к одному.
2. **Разобрали `Softmax`.** Превращает логиты в «уверенности» с суммой 1, считается по оси классов `dim=1`. Для самого ответа хватает `argmax` по логитам.
3. **Посчитали функции потерь.** `MSELoss` / `L1Loss` для регрессии, `CrossEntropyLoss` для многоклассовой классификации (на вход логиты, target — номера классов), BCE для бинарной.
4. **Посчитали метрики.** Accuracy для классификации и её слабые места (цена ошибки, несбалансированные классы). Для регрессии — округление и попадание в радиус.
5. **Сделали шаг оптимизатора.** `zero_grad → backward → step`, увидели, как lr влияет на сходимость: мало — медленно, много — скачет или расходится.
6. **Собрали сеть из видео 784 → 128 → 10** на `nn.Linear`, посчитали параметры (101 770) и разобрали ошибку формы.
7. **Отделили обучаемые параметры от гиперпараметров.**

## Зачем это знать

> Перед циклом обучения ещё раз глянь шпаргалку [`d.4.1 Полносвязные нейронные сети`](../../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.1 Полносвязные нейронные сети/d.4.1 Полносвязные нейронные сети.md) **§13** и словарь в конце файла: логит, softmax, CE, ReLU.

- **Это детали любой модели.** В видео 9–10 соберёшь их в класс `nn.Module`, в видео 11 — в цикл обучения: forward → loss → `zero_grad` → `backward` → `step` → метрика на val.
- **Неправильный loss или выход сети — тихие баги.** Softmax перед `CrossEntropyLoss` или `int`-метки в BCE не всегда падают с ошибкой: сеть просто учится хуже, и ты долго ищешь причину.
- **Размеры и число параметров** — защита от `shape mismatch` и способ понять, насколько тяжёлая модель и не переобучится ли она.
- **Для проекта:** FashionMNIST → сеть 784 → 128 → 10, `CrossEntropyLoss`, accuracy; квадраты → выход `(B, 2)`, `MSELoss`, метрика «попадание в радиус».
- **Для рекомендаций:** клик / не клик — бинарная задача: один логит на пару «пользователь–товар» и `BCEWithLogitsLoss`. Кликов обычно мало, поэтому accuracy там обманывает — как в 4.2.

## Шпаргалка: задача → выход → loss → метрика

| задача | выход последнего слоя | активация на выходе | loss | метрика |
|:---|:---|:---|:---|:---|
| многоклассовая (FashionMNIST) | логиты `(B, 10)` | нет — softmax внутри loss | `nn.CrossEntropyLoss()` | accuracy |
| бинарная (клик / не клик) | логит `(B,)` | нет — sigmoid внутри loss | `nn.BCEWithLogitsLoss()` | accuracy + цена ошибки |
| регрессия (центр квадрата) | `(B, 2)` | нет | `nn.MSELoss()` / `nn.L1Loss()` | попадание в радиус |

| инструмент | роль |
|:---|:---|
| `nn.ReLU()` / `nn.Sigmoid()` / `nn.Tanh()` | нелинейность между слоями |
| `nn.Softmax(dim=1)` | логиты → уверенности по классам, сумма 1 |
| `torch.optim.Adam(params, lr=0.001)` | оптимизатор; шаг: `zero_grad` → `backward` → `step` |
| `nn.Linear(in, out)` | полносвязный слой; параметров `in · out + out` |
| `sum(p.numel() for p in layer.parameters())` | число обучаемых параметров |

**Дальше:** видео 9 — классы для создания нейросетей (`nn.Module`, `nn.Sequential`). Там эти слои станут одной моделью.
